In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -r /content/drive/MyDrive/ml_projects/NLP_Text_Summarization/requirements.txt

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 75.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 80.5 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=67b37b14664e5d8356262be7ca14a7ce06744511f26c6ca8a18a70aa96750e7e
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score


In [3]:
!pip install -q transformers datasets peft accelerate sentencepiece evaluate rouge-score

In [4]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from datasets import load_from_disk
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)
from peft import (
    LoraConfig,
    get_peft_model,
    TaskType,
)
import evaluate

In [5]:
PROJECT_ROOT = Path(
    "/content/drive/MyDrive/ml_projects/NLP_Text_Summarization"
)

PROCESSED_DATA_DIR = PROJECT_ROOT / "artifacts/data/processed"
TOKENIZER_DIR = PROJECT_ROOT / "artifacts/tokenizer"

LORA_MODEL_DIR = PROJECT_ROOT / "artifacts/models/lora_pegasus"
LORA_RESULTS_DIR = PROJECT_ROOT / "artifacts/lora_results"

LORA_MODEL_DIR.mkdir(parents=True, exist_ok=True)
LORA_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

In [6]:
train_dataset = load_from_disk(
    PROCESSED_DATA_DIR / "tokenized_train"
)
validation_dataset = load_from_disk(
    PROCESSED_DATA_DIR / "tokenized_validation"
)
test_dataset = load_from_disk(
    PROCESSED_DATA_DIR / "tokenized_test"
)
print(train_dataset)
print(validation_dataset)
print(test_dataset)

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 14731
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 818
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 819
})


In [7]:
small_train = train_dataset.select(range(2000))
small_val = validation_dataset.select(range(200))

In [8]:
train_dataset=small_train
eval_dataset=small_val

In [9]:
MODEL_NAME = "google/pegasus-large"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/88.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/1.91M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

In [10]:
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

pytorch_model.bin:   0%|          | 0.00/2.28G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/680 [00:00<?, ?it/s]

Error during conversion: ReadTimeout('The read operation timed out')


model.safetensors:   0%|          | 0.00/2.28G [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
PegasusForConditionalGeneration LOAD REPORT from: google/pegasus-large
Key                                  | Status  | 
-------------------------------------+---------+-
model.decoder.embed_positions.weight | MISSING | 
model.encoder.embed_positions.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


generation_config.json:   0%|          | 0.00/260 [00:00<?, ?B/s]

In [11]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
print("Using device:", device)
model.to(device)

Using device: cuda


PegasusForConditionalGeneration(
  (model): PegasusModel(
    (shared): Embedding(96103, 1024, padding_idx=0)
    (encoder): PegasusEncoder(
      (embed_tokens): Embedding(96103, 1024, padding_idx=0)
      (embed_positions): PegasusSinusoidalPositionalEmbedding(1024, 1024)
      (layers): ModuleList(
        (0-15): 16 x PegasusEncoderLayer(
          (self_attn): PegasusAttention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (activation_fn): ReLU()
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=4096, out_features=1024, bias=True)
          (final_layer_no

In [12]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM,
)

In [14]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [15]:
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 3,145,728 || all params: 770,761,728 || trainable%: 0.4081


In [16]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
)

In [21]:
training_args = Seq2SeqTrainingArguments(

    output_dir=str(LORA_MODEL_DIR),
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_steps=100,
    logging_steps=100,
    eval_strategy="steps",
    eval_steps=100,
    save_steps=100,
    predict_with_generate=False,
    load_best_model_at_end=True,
    fp16=False,
    optim="adamw_torch",
    report_to="none",
)

In [22]:
model.print_trainable_parameters()

trainable params: 3,145,728 || all params: 770,761,728 || trainable%: 0.4081


In [23]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    data_collator=data_collator,
)

In [24]:
trainer.train()

Step,Training Loss,Validation Loss
100,61.472383,7.190705
200,57.841772,6.933125
300,55.728105,6.710222
400,54.430913,6.598065
500,53.782642,6.539053
600,53.275503,6.504890
700,52.747261,6.483959


TrainOutput(global_step=750, training_loss=55.43798990885416, metrics={'train_runtime': 2356.154, 'train_samples_per_second': 2.547, 'train_steps_per_second': 0.318, 'total_flos': 8726375301120000.0, 'train_loss': 55.43798990885416, 'epoch': 3.0})

In [25]:
model.save_pretrained(LORA_MODEL_DIR)
tokenizer.save_pretrained(LORA_MODEL_DIR)
print("LoRA adapters saved successfully.")

LoRA adapters saved successfully.


In [26]:
def generate_summary(dialogue_text):
    dialogue_text = "summarize: " + dialogue_text
    inputs = tokenizer(
        dialogue_text,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=512,
    )
    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }
    summary_ids = model.generate(
        **inputs,
        max_length=64,
        min_length=8,
        num_beams=8,
        repetition_penalty=2.5,
        no_repeat_ngram_size=3,
        length_penalty=1.0,
        early_stopping=True,
    )
    generated_summary = tokenizer.decode(
        summary_ids[0],
        skip_special_tokens=True,
    )
    return generated_summary

In [27]:
from datasets import load_dataset
raw_dataset = load_dataset("knkarthick/samsum")

sample_dialogue = raw_dataset["test"][0]["dialogue"]
reference_summary = raw_dataset["test"][0]["summary"]

generated_summary = generate_summary(sample_dialogue)

print("Generated Summary:\n")
print(generated_summary)

print("\nReference Summary:\n")
print(reference_summary)

README.md: 0.00B [00:00, ?B/s]

train.csv: 0.00B [00:00, ?B/s]

validation.csv: 0.00B [00:00, ?B/s]

test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/14731 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/818 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/819 [00:00<?, ? examples/s]

Generated Summary:

Hannah" SUMMER. her It number will

Reference Summary:

Hannah needs Betty's number but Amanda doesn't have it. She needs to contact Larry.


In [28]:
!pip install -q huggingface_hub

In [29]:
from huggingface_hub import notebook_login

notebook_login()

In [30]:
model.push_to_hub(
    "hyperchancellor07/pegasus_lora"
)

README.md:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   5%|4         |  574kB / 12.6MB            

CommitInfo(commit_url='https://huggingface.co/hyperchancellor07/pegasus_lora/commit/fc89ee56296f9f13589b60adb3c637b3df3268bb', commit_message='Upload model', commit_description='', oid='fc89ee56296f9f13589b60adb3c637b3df3268bb', pr_url=None, repo_url=RepoUrl('https://huggingface.co/hyperchancellor07/pegasus_lora', endpoint='https://huggingface.co', repo_type='model', repo_id='hyperchancellor07/pegasus_lora'), pr_revision=None, pr_num=None)

In [31]:
tokenizer.push_to_hub(
    "hyperchancellor07/pegasus_lora"
)

CommitInfo(commit_url='https://huggingface.co/hyperchancellor07/pegasus_lora/commit/d17531a98af5f67359a096162fe9ab1b6dc22287', commit_message='Upload tokenizer', commit_description='', oid='d17531a98af5f67359a096162fe9ab1b6dc22287', pr_url=None, repo_url=RepoUrl('https://huggingface.co/hyperchancellor07/pegasus_lora', endpoint='https://huggingface.co', repo_type='model', repo_id='hyperchancellor07/pegasus_lora'), pr_revision=None, pr_num=None)